In [1]:
import numpy as np

deg = {'SOFT': {'base': 90.0, 'd': 0.0844},
       'MEDIUM': {'base': 90.6, 'd': 0.0369},
       'HARD': {'base': 91.2, 'd': 0.0521}}
FUEL = -0.10
PIT_LOSS = 26.4          # green pit cost (measured)
PIT_SC   = 12.0          # pit cost under safety car (cited, Aguad)
N_LAPS   = 57
COMPOUNDS = ['SOFT', 'MEDIUM', 'HARD']

P_SC   = 0.0135          # your 2025 measurement: 54%/race -> ~1.35%/lap
SC_DUR = 4               # (legacy) unused; the chain now has 2 states
Q_END  = 0.25            # SC ends per lap (memoryless); E[dur]=1/q=4
SC_LAP_TIME = 140.0      # everyone crawls; same for all, so it doesn't distort choices

def lap_time(compound, tyre_age, lap_number):
    p = deg[compound]
    return p['base'] + p['d'] * tyre_age + FUEL * lap_number

In [2]:
def flag_transition(f):
    """Returns list of (next_flag, probability). The Markov chain."""
    if f == 0:                       # green now
        return [(0, 1 - P_SC), (1, P_SC)]
    else:                            # under SC: ends w.p. Q_END, else continues
        return [(1, 1 - Q_END), (0, Q_END)]

# visualize it as a matrix
states = list(range(2))     # 0=green, 1=under safety car
T = np.zeros((len(states), len(states)))
for f in states:
    for nf, p in flag_transition(f):
        T[f, nf] = p
print("Flag transition matrix (rows=now, cols=next):")
print(np.round(T, 4))

Flag transition matrix (rows=now, cols=next):
[[0.9865 0.0135]
 [0.25   0.75  ]]


In [3]:
NC, NA, NM, NF = 3, N_LAPS + 1, 2, 2
INF = float('inf')

# V[n, c, a, m, f]; terminal layer: legal only if two compounds used (m=1)
V   = np.full((N_LAPS + 2, NC, NA, NM, NF), INF)
POL = np.full((N_LAPS + 1, NC, NA, NM, NF), -1, dtype=int)   # -1=stay, 0/1/2=pit to that compound
V[N_LAPS + 1, :, :, 1, :] = 0.0          # finished having used 2 compounds: OK
                                          # m=0 stays INF -> disqualified

for n in range(N_LAPS, 0, -1):
    for ci, c in enumerate(COMPOUNDS):
        for a in range(NA):
            for m in range(NM):
                for f in range(NF):
                    best_val, best_act = INF, -1
                    # ---- action: STAY ----
                    if f == 0:                                   # green lap
                        if a + 1 < NA:
                            cost = lap_time(c, a + 1, n)
                            ev = sum(p * V[n+1, ci, a+1, m, nf]
                                     for nf, p in flag_transition(f))
                            if cost + ev < best_val:
                                best_val, best_act = cost + ev, -1
                    else:                                        # SC lap: no wear
                        cost = SC_LAP_TIME
                        ev = sum(p * V[n+1, ci, a, m, nf]
                                 for nf, p in flag_transition(f))
                        if cost + ev < best_val:
                            best_val, best_act = cost + ev, -1
                    # ---- action: PIT to compound x ----
                    for xi, x in enumerate(COMPOUNDS):
                        m2 = 1 if (xi != ci or m == 1) else 0
                        if f == 0:
                            cost = lap_time(x, 1, n) + PIT_LOSS
                            a2 = 1
                        else:
                            cost = SC_LAP_TIME + PIT_SC
                            a2 = 0
                        ev = sum(p * V[n+1, xi, a2, m2, nf]
                                 for nf, p in flag_transition(f))
                        if cost + ev < best_val:
                            best_val, best_act = cost + ev, xi
                    V[n, ci, a, m, f] = best_val
                    POL[n, ci, a, m, f] = best_act
    if n % 10 == 0: print(f"  solved back to lap {n}")

# best starting compound (fresh tires, rule unmet, green flag)
starts = {c: V[1, i, 0, 0, 0] for i, c in enumerate(COMPOUNDS)}
print("\nExpected race time by starting compound:", {k: round(v,1) for k,v in starts.items()})

  solved back to lap 50
  solved back to lap 40
  solved back to lap 30
  solved back to lap 20
  solved back to lap 10

Expected race time by starting compound: {'SOFT': 5194.3, 'MEDIUM': 5194.1, 'HARD': 5210.2}


In [4]:
def simulate(policy, flag_sequence, start_comp='SOFT'):
    """Run one race following the policy under a given flag sequence."""
    ci = COMPOUNDS.index(start_comp)
    a, m, total = 0, 0, 0.0
    pits = []
    for n in range(1, N_LAPS + 1):
        f = flag_sequence[n - 1]
        act = policy[n, ci, a, m, f]
        if act == -1:                            # stay
            if f == 0:
                a += 1; total += lap_time(COMPOUNDS[ci], a, n)
            else:
                total += SC_LAP_TIME
        else:                                    # pit
            pits.append((n, COMPOUNDS[act], 'SC' if f > 0 else 'green'))
            if act != ci or m == 1: m = 1
            ci = act
            if f == 0:
                a = 1; total += lap_time(COMPOUNDS[ci], a, n) + PIT_LOSS
            else:
                a = 0; total += SC_LAP_TIME + PIT_SC
    return total, pits

all_green = [0] * N_LAPS
t, pits = simulate(POL, all_green, 'SOFT')
print(f"Green-only race: time {t:.1f}s, pits: {pits}")
print(f"Model 0 (deterministic) pitted on lap 22 — did the SDP wait longer?")

Green-only race: time 5056.7s, pits: [(24, 'MEDIUM', 'green')]
Model 0 (deterministic) pitted on lap 22 — did the SDP wait longer?


In [5]:
rng = np.random.default_rng(42)

def random_flags():
    seq, f = [], 0
    for _ in range(N_LAPS):
        seq.append(f)
        if f == 0:
            f = 1 if rng.random() < P_SC else 0
        else:
            f = 0 if rng.random() < Q_END else 1
    return seq

def fixed_plan_time(flag_sequence):
    """Model 0's plan: soft 22 laps, pit lap 22, medium to the end — regardless of flags."""
    ci, a, total = 0, 0, 0.0                       # start SOFT
    for n in range(1, N_LAPS + 1):
        f = flag_sequence[n - 1]
        pit_now = (n == 22)
        if pit_now:
            ci = 1                                  # to MEDIUM
            if f == 0: a = 1; total += lap_time('MEDIUM', a, n) + PIT_LOSS
            else:      a = 0; total += SC_LAP_TIME + PIT_SC
        else:
            if f == 0: a += 1; total += lap_time(COMPOUNDS[ci], a, n)
            else:      total += SC_LAP_TIME
    return total

results_pol, results_fix = [], []
for _ in range(1000):
    flags = random_flags()
    tp, _ = simulate(POL, flags, 'SOFT')
    tf = fixed_plan_time(flags)
    results_pol.append(tp); results_fix.append(tf)

results_pol, results_fix = np.array(results_pol), np.array(results_fix)
print(f"Fixed plan   mean: {results_fix.mean():.1f}s")
print(f"SDP policy   mean: {results_pol.mean():.1f}s")
print(f"Average advantage: {(results_fix - results_pol).mean():.2f}s per race")
print(f"Policy wins or ties in {(results_pol <= results_fix).mean():.0%} of races")

Fixed plan   mean: 5200.2s
SDP policy   mean: 5198.5s
Average advantage: 1.75s per race
Policy wins or ties in 72% of races


In [6]:
flags = [0]*19 + [1,1,1,1] + [0]*(N_LAPS-23)
t, pits = simulate(POL, flags, 'SOFT')
print(f"Race with SC on laps 20-23: time {t:.1f}s, pits: {pits}")

Race with SC on laps 20-23: time 5243.7s, pits: [(20, 'MEDIUM', 'SC')]
